# 03 — Harmonic Balance

**Topic**: Fourier ansatz, HB residual derivation, AFT pipeline, Newton step visualised.

**Reference**: Krack & Gross (2019) §2, §3, Appendix C

**Estimated runtime**: < 10 seconds

## Theory

**Fourier ansatz** (K&G §2.1, eq. 2.3): assume periodic steady-state response with $H$ harmonics:

$$q(t) = Q_0 + \sum_{h=1}^{H} \left[ Q_{ch}\cos(h\Omega t) + Q_{sh}\sin(h\Omega t) \right] \tag{1}$$

The full coefficient vector is $\mathbf{Q} \in \mathbb{R}^{n_{dof}(2H+1)}$.

**HB residual** (K&G §2.2): substituting eq. (1) into the EoM and Galerkin-projecting onto $\{1, \cos(h\Omega t), \sin(h\Omega t)\}$ yields:

$$R(\mathbf{Q}, \Omega) = Z(\Omega)\mathbf{Q} + \hat{F}_{nl}(\mathbf{Q}, \Omega) - \hat{F}_{ext} = 0 \tag{2}$$

where $Z(\Omega)$ is the block-diagonal **dynamic stiffness matrix** and $\hat{F}_{nl}$ contains the Fourier coefficients of the nonlinear forces.

**AFT pipeline** (K&G §2.3):

```
Q (Fourier) ──IFFT──> q(t) (time domain) ──f_nl──> f_nl(t) ──FFT──> F_nl (Fourier)
```

The AFT avoids computing Fourier integrals analytically — only `eval(q, dq)` is needed.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

from nlvib.nonlinearities.elements import cubic_spring, unilateral_spring
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.solvers.harmonic_balance import hb_residual

# Duffing oscillator parameters (K&G §5.1)
m, d, k, k3, F = 1.0, 0.02, 1.0, 0.5, 0.1

system = SingleMassOscillator(m=m, d=d, k=k)
system.add_nonlinear_element(cubic_spring(k3=k3, dof_index=0))

print("Duffing oscillator ready.")

In [ ]:
# Visualise the Fourier coefficient layout for H=3 harmonics, n_dof=1
# Q = [Q_0 | Q_c1 Q_s1 | Q_c2 Q_s2 | Q_c3 Q_s3]
#         0     1    2     3    4     5    6

H     = 3
omega = 1.02  # near resonance
excitation = {'dof': 0, 'amplitude': F, 'harmonic': 1}

# Solve for Q at this frequency using Newton
Q = np.zeros(2*H + 1)
Q[1] = F / abs(-(omega**2)*m + k + 1j*omega*d)
for _ in range(30):
    R, J = hb_residual(Q, omega, system, H, excitation)
    if np.linalg.norm(R) < 1e-12:
        break
    Q = Q + np.linalg.solve(J, -R)

# Compute harmonic amplitudes |Q_h| = sqrt(Q_ch^2 + Q_sh^2)
labels = ['DC']
amps   = [abs(Q[0])]
for h in range(1, H+1):
    labels.append(f'H{h}')
    amps.append(np.sqrt(Q[2*h-1]**2 + Q[2*h]**2))

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(labels, amps, color=['tab:gray'] + ['tab:blue']*H)
ax.set_ylabel('Amplitude $|Q_h|$')
ax.set_title(f'Fourier coefficient layout — Duffing at $\\Omega={omega}$ rad/s, H={H}')
ax.set_yscale('log')
for bar, amp in zip(bars, amps):
    if amp > 1e-15:
        ax.text(bar.get_x() + bar.get_width()/2, amp * 1.5, f'{amp:.2e}',
                ha='center', va='bottom', fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
fig

In [ ]:
# Evaluate residual at the converged solution — should be ~0
R_check, _ = hb_residual(Q, omega, system, H, excitation)
print(f"||R|| at converged solution = {np.linalg.norm(R_check):.4e}  (should be < 1e-10)")

# Show effect of n_harmonics on residual quality
print("\nEffect of n_harmonics on residual at omega=1.02 (unilateral spring — rich harmonics):")

sys_uni = SingleMassOscillator(m=1.0, d=0.05, k=1.0)
sys_uni.add_nonlinear_element(unilateral_spring(k=1.0, gap=0.05, dof_index=0))

for H_test in [1, 3, 5, 7, 9]:  # ← try changing this
    Q_t = np.zeros(2*H_test + 1)
    lin_a = F / abs(-(omega**2)*1.0 + 1.0 + 1j*omega*0.05)
    Q_t[1] = float(lin_a)
    for _ in range(50):
        R_t, J_t = hb_residual(Q_t, omega, sys_uni, H_test, excitation)
        if np.linalg.norm(R_t) < 1e-12:
            break
        try:
            Q_t = Q_t + np.linalg.solve(J_t, -R_t)
        except np.linalg.LinAlgError:
            break
    amp_H = np.sqrt(Q_t[1]**2 + Q_t[2]**2) if H_test >= 1 else 0.0
    print(f"  H={H_test:2d}: amp(H1)={amp_H:.6f}, ||R||={np.linalg.norm(R_t):.2e}")

In [ ]:
# Visualise one Newton step: compute R and J, take step, show ||R|| drops

H_vis = 3
omega_vis = 1.05
excitation = {'dof': 0, 'amplitude': F, 'harmonic': 1}

# Perturbed initial guess
Q_init = np.zeros(2*H_vis + 1)
Q_init[1] = 0.3  # deliberately wrong

norms_before = []
norms_after  = []
Q_track = Q_init.copy()
for step in range(12):
    R0, J0 = hb_residual(Q_track, omega_vis, system, H_vis, excitation)
    norms_before.append(np.linalg.norm(R0))
    dQ = np.linalg.solve(J0, -R0)
    Q_track = Q_track + dQ
    R1, _ = hb_residual(Q_track, omega_vis, system, H_vis, excitation)
    norms_after.append(np.linalg.norm(R1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(norms_before, 'o-', label='||R|| before step', color='tab:orange')
ax.semilogy(norms_after,  's--', label='||R|| after step',  color='tab:blue')
ax.set_xlabel('Newton iteration')
ax.set_ylabel('Residual norm $\\|R\\|_2$')
ax.set_title(f'Newton convergence — Duffing HB, $\\Omega={omega_vis}$ rad/s, H={H_vis}')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig

## Key Takeaways

- The Fourier coefficient vector $\mathbf{Q}$ has layout: `[DC, cos1, sin1, cos2, sin2, ...]`.
- Higher harmonics decay rapidly for smooth nonlinearities (cubic spring), but slowly for impact/contact (unilateral spring).
- Newton convergence is quadratic once near the solution — 5–8 iterations typically suffice.
- The AFT pipeline is black-box from the user perspective: only `eval(q, dq)` is needed per nonlinear element.